# Importaciones

In [1]:
include("dependencies.jl")
include("helpers.jl")
include("wrappers.jl")
include("Pipeline_basics.ipynb")

# Cargamos los datos preprocesados
JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval
n_features = 561;

LoadError: LoadError: syntax: { } vector syntax is discontinued around c:\Users\Pc\Desktop\Uni\Tercero\MAAA\MAAA-I\Pipeline_basics.ipynb:1
in expression starting at c:\Users\Pc\Desktop\Uni\Tercero\MAAA\MAAA-I\Pipeline_basics.ipynb:1

# Modelos de Ensemble (30%)

In [ ]:
# ==============================================================================
# CONFIGURACIÓN DE LOS EXPERIMENTOS DE ENSEMBLE
# ==============================================================================

# 1. Configuración de Filtros (Solo probaremos sin filtrar de momento)
dic_filtros_basico = Dict(
    "Sin_Filtrado" => nothing
)

# 2. Configuración de Reducciones (PCA 95% y Sin Reducción)
dic_reducciones_ensemble = Dict(
    "Sin_Reduccion" => IdentityTransformer(), # O 'nothing' si prefieres
    "PCA_95"        => PCA(pratio=0.95)
)

# 3. Definición de Modelos Base para los Ensembles
# ----------------------------------------------------------------
# Base para Bagging: KNN
knn_base = KNNClassifier(K=5)

# Base para AdaBoost: SVM Lineal 
# Usamos SGDClassifier con loss="hinge" que ES un SVM lineal, 
# pero compatible con el AdaBoost de ScikitLearn.
svm_base = SKSGDClassifier(
    loss         = "hinge",   # Hinge loss = comportamiento de SVM
    penalty      = "l2",      # Regularización estándar
    alpha        = 0.0001,    
    random_state = SEED       # Para reproducibilidad
)

# 4. Diccionario de Modelos de Ensemble
dic_modelos_ensemble = Dict(
    # --- Bagging (KNN) ---
    "Bagging_KNN_10" => EnsembleModel(
        model = knn_base,
        n     = 10
    ),
    "Bagging_KNN_50" => EnsembleModel(
        model = knn_base,
        n     = 50
    ),

    # --- AdaBoost (SVM Lineal) ---
    "AdaBoost_SVM" => AdaBoostClassifier(
        base_estimator = svm_base,
        n_estimators   = 5,
        algorithm      = "SAMME" # Obligatorio para SVM/Hinge loss
    ),

    # --- EvoTrees (Gradient Boosting) ---
    "EvoTree_50" => EvoTreeClassifier(
        nrounds = 50, 
        eta     = 0.2
    ),
    "EvoTree_100" => EvoTreeClassifier(
        nrounds = 100, 
        eta     = 0.2
    )
)

UndefVarError: UndefVarError: `IdentityTransformer` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
run_experiment(
    dic_filtros_basico, 
    dic_reducciones_ensemble, 
    dic_modelos_ensemble, 
    "resultados_ensembles_bagging.csv" # Guardamos en un archivo nuevo
)